# Hand validation of rule detection and checkers

1. Do the rule regexes catch real rules? (print random matching lines)
2. Export every `package_manager` violation to `package_manager_violations.csv` for hand checking

`validation_sample.csv` / `validation_sample_v2.csv` were labelled by hand; results are in `RESULTS.md`.

In [1]:
import re

import pandas as pd

from ess_msr.data import ROOT, context_text, read_file
from ess_msr.rules import LOCKFILE_MANAGER, RULE_PATTERNS, load_labeled, required_managers

prs, pr_rule = load_labeled()
under_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])

23422 of 71677 agent PRs in 1326 repos had a context file


590 of them have no file info


## 1. Sanity check: do the regexes catch real rules?

Print random lines that matched each rule. If many are not real rules, tighten the pattern in `rules.py` and rerun.

In [2]:
def matching_lines(rule, n=20):
    hashes = under_rule.loc[under_rule.rules == rule, "context_hashes"].explode().unique()

    lines = []
    for file_hash in hashes:
        for line in read_file(file_hash).splitlines():
            if re.search(RULE_PATTERNS[rule], line, re.IGNORECASE):
                lines.append(line.strip())

    lines = pd.Series(lines).drop_duplicates()
    return lines.sample(min(n, len(lines)), random_state=0)


for rule in RULE_PATTERNS:
    print(f"\n===== {rule} =====")
    for line in matching_lines(rule):
        print("-", line[:200])


===== commit_format =====
- - Commitizen for conventional commits
- - **Conventional Commits**: `feat:`, `fix:`, `docs:`, `refactor:`, `chore:`
- - Use Conventional Commits where possible: `feat:`, `fix:`, `chore:`, `refactor:`, etc. Keep messages imperative and scoped (e.g., `feat(api): add chat stream`).
- - Conventional commits enforced by hook: `type(scope): subject` ≤ 50 chars; types: `feat|fix|docs|dx|style|refactor|perf|test|workflow|build|ci|chore|types|wip|release`.
- - Ensure the PR title is descriptive and follows the conventional commit format
- - Use Conventional Commits: `type(scope): description` in imperative mood, ≤72 characters, no trailing period.
- - Follow the [Conventional Commits](https://www.conventionalcommits.org/) specification for all commit messages
- - **Commit Messages**: Use clear, descriptive commit messages following the Conventional Commits format:
- - Follow [Conventional Commits 1.0.0](https://www.conventionalcommits.org/en/v1.0.0/)
- - Use convent

- - Always add test fixtures when modifying detection patterns
- - **Always add tests for new features**
- **Every new feature MUST include integration tests that validate ACTUAL platform behavior.** Integration tests are not optional - they are required because they:
- 2. **Write tests first** for new components and hooks before implementation
- 3. Add/update tests for new features
- - ALL new features MUST include comprehensive unit tests
- 5. Write tests for new functionality
- - Write go unit tests whenever possible
- **MANDATORY REQUIREMENT:** Every code review must include a test suggestion using the Quick Reference Template provided below. This is required for all pull request reviews to ensure consistent test c
- Stream indicators must include performance tests in the `tests/performance` project for high-frequency scenarios:
- - Add tests for new API endpoints and features
- - Always include test cases for critical paths of the application.
- - Write unit tests for new function

- Run `pnpm install` to hydrate the workspace (Node.js ≥ 20.19.0 is required). Build all packages with `pnpm run build`. Execute `pnpm run test` for the Vitest suite or `pnpm run test:cov` to generate c
- **Package Manager**: Use Yarn exclusively (configured in .yarnrc.yml)
- - Use `pnpm add` instead of `npm install <package>`
- 5. Run `pnpm install` to update dependencies
- - Use Yarn (not npm) for package management
- - do not apply any migrations, just use `pnpm sync`
- - Use `pnpm` as the package manager
- - Use `pnpm add` for adding dependencies instead of `npm install` or `yarn add`
- ## Use pnpm Only
- - **Use Bun exclusively.** Run installs with `bun install`, scripts with `bun run <script>`, add deps via `bun add` / `bun add -d`.
- - **Package Manager**: pnpm (v10+)
- - [uv.lock](mdc:uv.lock) - Locked dependencies using uv package manager
- - Yarn package manager (v1.22.19+)
- Only use pnpm
- - When suggesting dependency management, use `uv sync` instead of `pip install -r req

## 2. Hand check: package_manager violations

Open each PR link, look at the added lockfile, and fill the `correct` column in the CSV with `yes` / `no`.

The CSV is only written if it does not exist yet, so hand labels are never overwritten. Delete it to regenerate.

In [3]:
def rule_line(hashes):
    for line in context_text(hashes).splitlines():
        if re.search(RULE_PATTERNS["package_manager"], line, re.IGNORECASE):
            return line.strip()


violations = pr_rule[(pr_rule.rules == "package_manager") & (pr_rule.label == "violated")]

to_check = pd.DataFrame({
    "pr_url": violations.html_url,
    "agent": violations.agent,
    "rule_line": violations.context_hashes.apply(rule_line),
    "required": violations.context_hashes.apply(lambda h: sorted(required_managers(context_text(h)))),
    "added_lockfiles": violations.added_files.apply(lambda files: [f for f in files if f.split("/")[-1] in LOCKFILE_MANAGER]),
    "correct": "",
})
csv_path = ROOT / "package_manager_violations.csv"
if csv_path.exists():
    print(csv_path.name, "already exists, not overwritten")
else:
    to_check.to_csv(csv_path, index=False)

pd.set_option("display.max_colwidth", 120)
to_check

package_manager_violations.csv already exists, not overwritten


,pr_url,agent,rule_line,required,added_lockfiles,correct
700,https://github.com/ecomfe/vue-echarts/pull/923,Copilot,Use `pnpm install` to set up dependencies. `pnpm dev` serves the demo playground at `http://localhost:5173` for inte...,[pnpm],[package-lock.json],
2216,https://github.com/feldera/feldera/pull/4546,Copilot,- **Package Manager**: Bun (not npm),[bun],[web-console/package-lock.json],
2254,https://github.com/jesec/flood/pull/933,Copilot,# Package Manager: pnpm (v9.7.0) is the project's package manager,[pnpm],[package-lock.json],
2577,https://github.com/microsoft/typespec/pull/7661,Copilot,- Use `pnpm` as the package manager,[pnpm],[packages/http-client-java/package-lock.json],
3483,https://github.com/ast-grep/ast-grep-vscode/pull/495,Copilot,- **Node.js** and **pnpm**: Package manager for dependencies,[pnpm],[package-lock.json],
5674,https://github.com/demokratie-live/democracy-client/pull/1652,Copilot,- Yarn package manager (v1.22.19+),[yarn],[pnpm-lock.yaml],
6395,https://github.com/usememos/memos/pull/4814,Copilot,- **Package Manager:** pnpm,[pnpm],[web/package-lock.json],
7495,https://github.com/svelteplot/svelteplot/pull/182,Copilot,- **PNPM**: Package manager (preferred over npm/yarn),[pnpm],[package-lock.json],
7777,https://github.com/Azure/azure-sdk-for-js/pull/35787,Copilot,- Use pnpm for package management (never npm or yarn),[pnpm],[eng/common/tsp-client/package-lock.json],
8579,https://github.com/Fraunhofer-AISEC/cpg/pull/2386,Copilot,- **Always use `pnpm` for Node.js/JavaScript/TypeScript projects**,[pnpm],[codyze-console/src/main/webapp/package-lock.json],
